In [1]:
# pip install FlagEmbedding

In [1]:
import os
import re
import time
from typing import Any
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from pinecone import Pinecone, ServerlessSpec
from pinecone_text.sparse import BM25Encoder
from sentence_transformers import CrossEncoder
from tqdm.notebook import tqdm


class RAG_service:

    def __init__(
            self,
            index_name: str,
            api_key: str,
            cloud: str,
            region: str,
            dimension: int = 1024
    ):
        """
        创建初始化类
        包含:
        初始化Pinecone
        md文档分割工具
        递归分块(段落 句子)
        BGA向量化
        BM25稀疏矩阵向量化
        BAAI重排序模型

        :param index_name:
        :param api_key:
        :param cloud:
        :param region:
        :param dimension:
        """
        self.index_name = index_name
        self.api_key = api_key
        self.cloud = cloud
        self.region = region
        self.dimension = dimension
        self.pc = Pinecone(api_key=api_key)
        self.index = self.pc.Index(self.index_name)

        # 初始化工具
        headers_to_split_on = [("#", "Header_1")]

        # md文档分割
        self.md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
        # 段落句子分割
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=512,
            chunk_overlap=50,
            separators=["\n\n", "\n", ".", "；", " "],
            add_start_index=True
        )

        # 重排序模型
        self.reranker = CrossEncoder('BAAI/bge-reranker-large', max_length=512)

        # 稀疏向量
        self.bm25 = BM25Encoder().load("bm25_law_params.json")

        # 密集向量
        model_name = "BAAI/bge-large-zh-v1.5"
        self.embeddings = HuggingFaceEmbeddings(
            model_name=model_name,
            encode_kwargs={'normalize_embeddings': True}
        )

    def create_index(self, wait_for_completion: bool = True) -> bool:
        """
        创建索引
        创建混合索引
        指定的索引方式
        :param wait_for_completion:
        :return:
        """
        self.pc = Pinecone(api_key=self.api_key)

        # 混合索引的强制要求：metric 必须为 dotproduct,vector_type 为 dense
        target_metric = "dotproduct"

        # 检查索引是否存在
        if not self.pc.has_index(self.index_name):
            print(f"正在创建混合检索索引: {self.index_name}...")
            self.pc.create_index(
                name=self.index_name,
                dimension=self.dimension,
                metric=target_metric,
                spec=ServerlessSpec(cloud=self.cloud, region=self.region),
                vector_type="dense",
            )
        else:
            print(f"索引 '{self.index_name}' 已存在.")
        # 等待索引就绪 异步处理
        if wait_for_completion:
            while not self.pc.describe_index(self.index_name).status.get("ready", False):
                time.sleep(2)
            print(f"索引 '{self.index_name}' 已就绪.")

        self.index = self.pc.Index(self.index_name)
        print(f"索引 '{self.index_name}' 已就绪.")

        return True

    def get_index_stats(self):
        """
        获取索引是否创建成功
        测试索引是否创建成功并打印统计信息
        :return: 是否创建成功
        """
        stats = self.index.describe_index_stats()
        print("当前索引的状态: ", stats)
        return True

    def get_Documents(self, file_path: str) -> list[Any] | None:
        """
        添加文本到数据库中需要:
        加载文本,
        文本分块,

        具体实施:
        (正则表达式)
        将清洗好的文件加载
        以每一个案例为单位
        先提取元数据
        (分块)
        从[基本案情]到最后的内容提取
        以句子为单位,合成一大段

        数据清洗:
        metadata:{year,case_number,case_cause,chunk_text}


        :param file_path:str
        :return 符合输入格式的列表
        """

        loader = TextLoader(
            file_path,
            encoding="utf-8"
        )

        # 插入Pinecone数据容器
        Pinecone_records = []

        # 默认读取的为Document形式
        documents = loader.load()

        # 按一级标题分割成多个案例
        articles = self.md_splitter.split_text(documents[0].page_content)

        # 获取元数据和正文内容
        for DocuID, article in enumerate(articles):
            print(f"第{DocuID}篇文章获取中...")
            # 元数据容器
            metadata = {}

            # 提取年份：假设文件名或路径包含 202X
            year_match = re.search(r"20\d{2}", article.page_content)
            metadata["year"] = year_match.group(0) if year_match else "Unknown"

            # 提取裁判书字号：匹配如（2023）最高法民终...号
            case_num_pattern = r'裁判书字号[\s\\n]+((?:(?!裁判书字号)[\s\S])+?法院[\s\S]+?书)'
            case_num_match = re.search(case_num_pattern, article.page_content)

            metadata["case_number"] = case_num_match.group(1).strip() if case_num_match else "未识别"

            # 提取案由：通常在字号之后,或者是特定的段落
            case_cause_pattern = r"案由[:：]\s*([\u4e00-\u9fa5]+)"
            cause_match = re.search(case_cause_pattern, article.page_content)

            metadata["case_cause"] = cause_match.group(1) if cause_match else "通用"

            # 最终的metadata示例: 'metadata': {'case_cause': ,'case_number': , 'chunk_index': 2,'chunk_text': }

            # 提取基本案情
            facts_pattern = r'【基本案情】\s*([\s\S]+?)(?=\n【|$)'
            facts_match = re.search(facts_pattern, article.page_content)

            if not facts_match:
                continue
            # 获取捕获组内容（不含【基本案情】）
            raw_content = facts_match.group(1)
            # 去除空格、换行、制表符等所有空白字符,以及 # 符号
            facts_cleaned = re.sub(r'\n+', '\n', raw_content).strip()  # 去除所有空白（空格、换行等）
            facts_cleaned = facts_cleaned.replace('#', '')  # 去除所有 # 字符

            print("元数据和原文解析完毕...")

            chunks = self.text_splitter.split_text(facts_cleaned)

            for i, chunk in enumerate(chunks):
                print(f"第{i}个记录创建中...")
                # 独有的
                record_id = f"annualCases{metadata['year']}_Docu{DocuID}_chunk{i}"

                # 密集向量
                dense_vector = self.embeddings.embed_query(chunk)

                # 返回 {"indices": [...], "values": [...]}
                sparse_vector = self.bm25.encode_documents(chunk)

                """
                添加内容: id 向量数据 元数据:{年份 判决书 案由 文档切片}
                符合Pinecone的输入格式
                """
                record = {
                    "id": record_id,
                    "values": dense_vector,  # 稠密向量列表
                    "sparse_values": sparse_vector,  # 稀疏向量字典
                    "metadata": {
                        **metadata,
                        "chunk_index": i,
                        "chunk_text": chunk,
                    }
                }
                Pinecone_records.append(record)

                print("记录插入完毕")

        return Pinecone_records

    def add_document(self, Pinecone_records, namespace: str, ):
        """
        添加分块好的文本到数据库中
        :param Pinecone_records:
        :param namespace:
        :return:
        """
        # 分批上传,每批最多 50 条向量
        batch_size = 50
        total = len(Pinecone_records)

        for i in range(0, total, batch_size):
            batch = Pinecone_records[i:i + batch_size]
            self.index.upsert(vectors=batch, namespace=namespace)
            uploaded = min(i + batch_size, total)
            print(f"已上传 {uploaded}/{total} 条记录")

        print(self.get_index_stats())

        return True

    def search_withDenseSparse(
            self,
            query: str,
            namespace: str,
            top_k: int = 50,
            rerank_top_n: int = 10,
            alpha: float = 0.5
    ) -> list:
        """
        分别获取问题的稀疏和密集向量化矩阵
        双路查询
        对结果和文字重排序
        分别检索的向量对文本意思没有关联
        交叉编码器同时接收查询‑文档对作为输入
        通过 Transformer 的全注意力机制（Self‑Attention）让查询和文档的每个词充分交互
        最终输出一个相关性分数
        :param query:
        :param namespace:
        :param top_k:
        :param rerank_top_n:
        :param alpha:
        :return:
        """

        progress = tqdm(total=400, desc="检索流程", unit="step")

        progress.set_description("生成查询向量")

        dense_vec = self.embeddings.embed_query(query)
        sparse_vec = self.bm25.encode_queries(query)

        for _ in range(100):
            time.sleep(0.01)
            progress.update(1)

        # 步骤2：混合召回
        progress.set_description("混合召回中")
        results = self.index.query(
            vector=dense_vec,
            sparse_vector=sparse_vec,
            alpha=alpha,
            namespace=namespace,
            top_k=top_k,
            include_metadata=True
        )
        matches = results.matches

        for _ in range(100):
            time.sleep(0.01)
            progress.update(1)

        if not matches:
            progress.close()
            print("未检索到任何结果")
            return []

        # 步骤3：提取文本对
        progress.set_description("提取文本对")

        texts = [m.metadata['chunk_text'] for m in matches]

        pairs = [[query, t] for t in texts]

        for _ in range(100):
            time.sleep(0.01)
            progress.update(1)

        # 步骤4：重排序
        progress.set_description("重排序中")
        scores = self.reranker.predict(pairs)
        for match, score in zip(matches, scores):
            match.rerank_score = score
        reranked = sorted(matches, key=lambda x: x.rerank_score, reverse=True)

        for _ in range(100):
            time.sleep(0.01)
            progress.update(1)
        progress.close()

        return reranked[:rerank_top_n]






In [2]:

load_dotenv()

# 实例化测试
service = RAG_service(
    index_name="pinecone-test-lawapp",
    api_key=os.getenv("PINECONE_API_KEY"),
    cloud="aws",
    region="us-east-1",
)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:


Pinecone_records = service.get_Documents("../MarkDownFiles/2023年度案例_test.md")

print("清晰后的结果为:", Pinecone_records[:1000])


第0篇文章获取中...
元数据和原文解析完毕...
第0个记录创建中...
记录插入完毕
第1个记录创建中...
记录插入完毕
第2个记录创建中...
记录插入完毕
第3个记录创建中...
记录插入完毕
第4个记录创建中...
记录插入完毕
第5个记录创建中...
记录插入完毕
第6个记录创建中...
记录插入完毕
第1篇文章获取中...
元数据和原文解析完毕...
第0个记录创建中...
记录插入完毕
第1个记录创建中...
记录插入完毕
第2个记录创建中...
记录插入完毕
第3个记录创建中...
记录插入完毕
第4个记录创建中...
记录插入完毕
第5个记录创建中...
记录插入完毕
第6个记录创建中...
记录插入完毕
第7个记录创建中...
记录插入完毕
第8个记录创建中...
记录插入完毕
第2篇文章获取中...
元数据和原文解析完毕...
第0个记录创建中...
记录插入完毕
第1个记录创建中...
记录插入完毕
第2个记录创建中...
记录插入完毕
第3个记录创建中...
记录插入完毕
第4个记录创建中...
记录插入完毕
第5个记录创建中...
记录插入完毕
第6个记录创建中...
记录插入完毕
第7个记录创建中...
记录插入完毕
第8个记录创建中...
记录插入完毕
第9个记录创建中...
记录插入完毕
第10个记录创建中...
记录插入完毕
第11个记录创建中...
记录插入完毕
第12个记录创建中...
记录插入完毕
第3篇文章获取中...
元数据和原文解析完毕...
第0个记录创建中...
记录插入完毕
第1个记录创建中...
记录插入完毕
第2个记录创建中...
记录插入完毕
第3个记录创建中...
记录插入完毕
第4个记录创建中...
记录插入完毕
第5个记录创建中...
记录插入完毕
第6个记录创建中...
记录插入完毕
第7个记录创建中...
记录插入完毕
第8个记录创建中...
记录插入完毕
第9个记录创建中...
记录插入完毕
第10个记录创建中...
记录插入完毕
第4篇文章获取中...
元数据和原文解析完毕...
第0个记录创建中...
记录插入完毕
第1个记录创建中...
记录插入完毕
第2个记录创建中...
记录插入完毕
第3个记录创建中...
记录插入完毕
第4个记录创建中...
记录插入完毕
第5个记录创建中...

In [ ]:
# 分批次添加记录
service.add_document(Pinecone_records, "Law_test_namespace")

In [3]:
search_docus = service.search_withDenseSparse("离婚后彩礼能归还吗?", "Law_test_namespace", 10, 3, 0.7)
for docu in search_docus:
    # print(f"检索结果为:{docu.id},案由{docu.metadata["case_cause"]}")
    print(docu)
    print("\n")

检索流程:   0%|          | 0/400 [00:00<?, ?step/s]

{'id': 'Docu1_chunk5',
 'metadata': {'case_cause': '婚约财产纠纷',
              'case_number': '河北省遵化市人民法院(2017)冀0281民初第5922号民事判决书',
              'chunk_index': 5,
              'chunk_text': '〈中华人民共和国婚姻法〉若干问题的解释（二）》第十条规定，当事人请求返还按照习俗给付的彩礼的，如果查明属于以下情形，人民法院应当予以支持：（一）双方未办理结婚登记手续的；（二）双方办理结婚登记手续但确未共同生活的；（三）婚前给付并导致给付人生活困难的。适用前款第（二）、（三）项的规定，应当以双方离婚为条件。然而在相关的法律中缺少关于如何返还已给付的彩礼、返还的依据是什么、返还的比例具体是多少的规定。在审判过程中，应该从以下几个方面来考虑：  \n'
                            '1. 彩礼的认定 '
                            '。男女双方在交往期间，必然会互相有所赠与，对于这些赠与我们需要区分哪些属于彩礼，哪些仅仅是赠与 '
                            '。与当地的生活水平进行比对，给付对方财物数额较大的，应该被认定为彩礼，予以返还；而对于那些双方之间互赠的 '
                            '、价值较小的财物，则不能认定为彩礼，不需要返还 。',
              'year': '2017'},
 'rerank_score': 0.9974458,
 'score': 0.7116642,
 'values': []}


{'id': 'Docu4_chunk5',
 'metadata': {'case_cause': '婚约财产纠纷',
              'case_number': '江苏省南通市中级人民法院(2017)苏06民终第1330号民事判决书',
              'chunk_index': 5,
              'chunk_text': '我国婚姻法司法解释虽然有关于彩礼返还的一些规定，但并没有以成文法的形式明确何

In [ ]:
RAG_Search_Results = service.search_withDenseSparse("离婚后彩礼能归还吗?", "Law_test_namespace", 20, 5, 0.3)


**检索结果的样例:**

```json
{
    'id': 'annualCases2022_Docu1_chunk7',
    'metadata': {
        'case_cause': '婚约财产纠纷',
        'case_number': '山东省德州市中级人民法院（2022）鲁14民终1192号民事判决书',
        'chunk_index': 7,
        'chunk_text': '……彩礼返还应当综合双方共同生活情况……',
        'year': '2022'
    },
    'rerank_score': 0.9838927,
    'score': 0.676428795,
    'values': []   # 通常为空或可忽略
}
```

**添加文件记录的样例:**
```json
{
'id': 'annualCases2022_Docu0_chunk0',
 'metadata':
{
'case_cause': '婚约财产纠纷',
 'case_number': '北京市第二中级人民法院（2022）京02民终9311号民事判决书', 'chunk_index': 0,
 'chunk_text': '刘某与赵某原系中学同学，于2019年2月确定恋爱关系，当时赵某在某985高校读研究生，刘某在其家乡工作。2019年11月28日赵某发现自己已怀孕。2019年12月29日，双方举办订婚仪式。2019年12月30日，刘某经家人向赵某转账 131400 元。2020 年 5 月 6 日，双方举办了婚礼，同年 7 月 20 日生育一女。此期间赵某研究生毕业，入职工作，此后赵某拒绝办理其与刘某的结婚登记手续。刘某主张双方恋爱期间，其与家人向赵某多次转账和购买贵重物品，要求赵某返还给其转账及花费的支出以及 131400 元彩礼共计 30 余万元。赵某不认可 131400 元在性质上属于彩礼，认为属于普通赠与，其他转账和购买的贵重物品也都属于赠与，不同意返还；并认为其拒绝登记结婚是因双方性格不合、缺乏感情。刘某认为自己与家人向赵某多次转账和购买贵重物品的行为，在赵某恋爱、怀孕至生育期间对其精心照顾，全力满足其要求的行为，均可证明自己及家人对赵某倍加珍惜和关爱，是赵某在入职后找借口拒绝登记结婚。
 【案件焦点】',
 'year': '2022'
  },
 'sparse_values': {'indices': [877881977, 3130549363,3557921014],
 'values': [0.6165115996132395,0.6165115996132395]},
'values': [-0.022952565923333168 ...... -0.0009091026149690151, 0.025131508708000183, -0.004482208285480738]
  }
}
```